# Obiettivo

In questa nuova pipeline non costruiamo un modello da zero, ma partiamo dalla versione migliorata già ottenuta nel progetto precedente per introdurre ulteriori miglioramenti mirati.

L’obiettivo è aumentare la capacità del modello di generalizzare meglio su immagini mai viste, ridurre l’overfitting e migliorare la distinzione tra classi che tendono a confondersi più facilmente, come **cat e dog**, **bird e deer**, oppure **automobile e truck**.

Per raggiungere questo obiettivo introdurremo quattro miglioramenti principali.

Il primo sarà il potenziamento della **data augmentation**, aggiungendo non solo trasformazioni geometriche leggere come flip, rotazioni e traslazioni, ma anche piccole variazioni di **luminosità**, **contrasto** e, se possibile, di **colore/saturazione**. Questo servirà a rendere il modello meno sensibile alle condizioni di luce e alle differenze cromatiche specifiche del training set.

Il secondo miglioramento sarà la sostituzione di **Flatten** con **GlobalAveragePooling2D**. Questa modifica permette di ridurre il numero di parametri nella parte finale della rete, limitando il rischio di overfitting e rendendo l’architettura più compatta e moderna.

Il terzo miglioramento sarà l’aggiunta di un **blocco convoluzionale aggiuntivo** nella parte più profonda della rete. In questo modo il modello potrà estrarre feature più ricche e più astratte, utili soprattutto per distinguere immagini simili tra loro.

Infine, dopo aver costruito questa nuova base architetturale, useremo **Optuna** per il tuning automatico degli iperparametri. L’idea non è cambiare casualmente il modello, ma cercare in modo sistematico la combinazione di parametri che porta alle migliori prestazioni in validazione.

L’ordine seguito sarà quindi questo:
1. costruzione della nuova CNN migliorata;
2. addestramento e prima valutazione della nuova architettura;
3. tuning automatico con Optuna;
4. confronto finale con il modello precedente.

In questo modo sarà possibile capire non solo **se** il modello migliora, ma anche **quali modifiche** hanno contribuito di più al risultato finale.

# Import librerie

Oltre agli strumenti già usati nella pipeline precedente, qui aggiungiamo anche i componenti che serviranno per:
- introdurre augmentation più ricca sulle immagini;
- costruire un’architettura più avanzata;
- tracciare grafici utili per interpretare il comportamento del modello;
- eseguire il tuning automatico degli iperparametri.

L’idea è quindi preparare fin da subito un ambiente completo, così da non dover aggiungere import in modo frammentato nelle sezioni successive.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import platform
import sys

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import tensorflow as tf
import keras

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

## Setup GPU

In [ ]:
# Setup GPU TensorFlow
print("Keras backend:", keras.backend.backend())
print("TensorFlow version:", tf.__version__)
print("Sistema:", platform.platform())
print("Python:", sys.version.split()[0])

gpus = tf.config.list_physical_devices("GPU")
print("GPU fisiche rilevate:", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        logical_gpus = tf.config.list_logical_devices("GPU")
        print("GPU logiche:", logical_gpus)
        print("Memory growth abilitata correttamente.")
    except RuntimeError as e:
        print("Errore durante il setup GPU:", e)
else:
    print("Nessuna GPU rilevata da TensorFlow.")

# 1. Caricamento del dataset

Poiché il dataset è lo stesso della pipeline precedente, questa fase non cambia dal punto di vista dei dati di partenza. Quello che cambia è il modo in cui useremo queste immagini nelle fasi successive, grazie a una pipeline di augmentation più ricca e a un modello più avanzato.

Facciamo anche una prima verifica delle dimensioni degli array, così da confermare che il caricamento sia avvenuto correttamente.

In [ ]:
# Caricamento del dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Nomi delle classi
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Controllo delle shape
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape :", x_test.shape)
print("y_test shape :", y_test.shape)
print("\nNumero classi:", len(class_names))
print("Classi:", class_names)

## 1.1. Visualizzazione di prova delle immagini

Prima di allenare il modello è utile osservare visivamente alcune immagini per capire la varietà del dataset e la difficoltà del compito.


In [ ]:
plt.figure(figsize=(15, 6))

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

for class_index, class_name in enumerate(class_names):
    img_index = np.where(y_train.squeeze() == class_index)[0][0]

    plt.subplot(2, 5, class_index + 1)
    plt.imshow(x_train[img_index])
    plt.title(f'Etichetta: {class_name}')
    plt.axis('off')

plt.tight_layout()
plt.show()

# 2. Preprocessing

Il preprocessing serve a preparare i dati in una forma più adatta all’addestramento della rete neurale.

Anche in questa pipeline manteniamo i passaggi fondamentali già usati in precedenza, perché restano corretti anche per il nuovo modello. In particolare:
- normalizziamo i pixel dividendo per 255, così i valori passano dall’intervallo **[0, 255]** all’intervallo **[0, 1]**;
- convertiamo le etichette in formato **one-hot encoding**, così il modello potrà produrre una probabilità per ciascuna delle 10 classi.

La normalizzazione è importante perché rende il training più stabile e facilita l’ottimizzazione.  
La codifica one-hot, invece, è necessaria per usare correttamente la classificazione multiclasse con output finale softmax.

In questa sezione facciamo anche un piccolo controllo visivo per mostrare che la normalizzazione non cambia il contenuto dell’immagine, ma solo la scala numerica dei pixel.

In [ ]:
# Conversione delle etichette in one-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Normalizzazione dei pixel
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Controllo  delle shape
print("x_train shape:", x_train.shape)
print("x_test shape :", x_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("Esempio label one-hot :", y_train[0])

# 3. Tuning automatico con Optuna

Introduciamo **Optuna**, una libreria per l’ottimizzazione automatica degli iperparametri.  
L’idea è semplice: invece di scegliere manualmente i parametri del modello, lasciamo che Optuna provi diverse combinazioni e misuri quali funzionano meglio sul validation set.

In questa fase non stiamo ancora cercando il modello finale definitivo, ma stiamo cercando la **migliore configurazione di partenza** per:
- numero di filtri nei blocchi convoluzionali;
- dropout dei vari blocchi;
- numero di neuroni del classificatore finale;
- learning rate.

Ogni configurazione viene addestrata per poche epoche, in modo da confrontare rapidamente le varie soluzioni.  
Alla fine, Optuna ci restituirà la combinazione di iperparametri più promettente, che useremo per costruire e addestrare il modello finale in modo completo.


In [ ]:
from sklearn.model_selection import train_test_split

# Split fisso per il tuning con Optuna
x_train_opt, x_val_opt, y_train_opt, y_val_opt = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print("x_train_opt shape:", x_train_opt.shape)
print("x_val_opt shape  :", x_val_opt.shape)
print("y_train_opt shape:", y_train_opt.shape)
print("y_val_opt shape  :", y_val_opt.shape)

## 3.1 Definizione del modello parametrico

Per usare Optuna dobbiamo costruire una funzione capace di generare il modello in modo dinamico.

In pratica, invece di fissare a mano tutti i valori, lasciamo che alcuni parametri vengano scelti automaticamente da Optuna all’interno di un certo intervallo.

In questa nuova versione rendiamo ottimizzabili:
- il numero di filtri nei blocchi convoluzionali;
- i tassi di dropout;
- la dimensione del classificatore finale;
- il learning rate.

La struttura generale della rete resta comunque coerente con la nostra nuova base:
- data augmentation;
- blocchi convoluzionali con Batch Normalization e Dropout;
- blocco convoluzionale aggiuntivo;
- GlobalAveragePooling2D al posto di Flatten.

In questo modo Optuna non costruisce modelli casuali, ma esplora varianti sensate della stessa architettura.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2

# Layer personalizzato per color space augmentation leggera
class ColorJitter(tf.keras.layers.Layer):
    def __init__(self, brightness=0.08, contrast=0.12, saturation=0.10, **kwargs):
        super().__init__(**kwargs)
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation

    def call(self, images, training=None):
        if training is False:
            return images

        images = tf.image.random_brightness(images, max_delta=self.brightness)
        images = tf.image.random_contrast(
            images,
            lower=1.0 - self.contrast,
            upper=1.0 + self.contrast
        )
        images = tf.image.random_saturation(
            images,
            lower=1.0 - self.saturation,
            upper=1.0 + self.saturation
        )
        images = tf.clip_by_value(images, 0.0, 1.0)
        return images

# Data augmentation migliorata
data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    ColorJitter(brightness=0.08, contrast=0.12, saturation=0.10)
], name="data_augmentation")


def build_model(trial):
    filters_1 = trial.suggest_categorical("filters_1", [32, 48, 64])
    filters_2 = trial.suggest_categorical("filters_2", [64, 96, 128])
    filters_3 = trial.suggest_categorical("filters_3", [128, 160, 192])
    filters_4 = trial.suggest_categorical("filters_4", [192, 224, 256])

    dropout_1 = trial.suggest_float("dropout_1", 0.20, 0.35)
    dropout_2 = trial.suggest_float("dropout_2", 0.25, 0.40)
    dropout_3 = trial.suggest_float("dropout_3", 0.30, 0.45)
    dropout_4 = trial.suggest_float("dropout_4", 0.35, 0.50)
    dropout_dense = trial.suggest_float("dropout_dense", 0.30, 0.50)

    dense_units = trial.suggest_categorical("dense_units", [64, 128, 192, 256])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)

    model = Sequential([
        Input(shape=(32, 32, 3)),

        # Data augmentation migliorata
        data_augmentation,

        # Blocco 1
        Conv2D(filters_1, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(filters_1, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(dropout_1),

        # Blocco 2
        Conv2D(filters_2, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(filters_2, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(dropout_2),

        # Blocco 3
        Conv2D(filters_3, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(filters_3, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(dropout_3),

        # Blocco 4 aggiuntivo
        Conv2D(filters_4, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(filters_4, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(dropout_4),

        # Parte finale compatta
        GlobalAveragePooling2D(),
        Dense(dense_units, activation="relu"),
        BatchNormalization(),
        Dropout(dropout_dense),
        Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

## 3.2 Funzione objective di Optuna

La funzione `objective` è il cuore del tuning automatico.

Per ogni trial, Optuna:
- sceglie una combinazione di iperparametri;
- costruisce un nuovo modello con quei valori;
- lo addestra per un numero limitato di epoche;
- misura la validation accuracy;
- usa questo risultato per decidere quali configurazioni esplorare dopo.

In questa fase il training deve essere più corto rispetto a quello finale, perché il suo scopo non è ottenere subito il miglior modello possibile, ma confrontare rapidamente molte configurazioni diverse.

Per questo usiamo:
- poche epoche;
- EarlyStopping;
- ReduceLROnPlateau;
- validation set fisso.

Il valore restituito sarà la **migliore validation accuracy** raggiunta nel trial.

In [ ]:
def objective(trial):
    keras.backend.clear_session()

    model = build_model(trial)

    early_stop_optuna = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    reduce_lr_optuna = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )

    history = model.fit(
        x_train_opt,
        y_train_opt,
        validation_data=(x_val_opt, y_val_opt),
        epochs=20,
        batch_size=128,
        callbacks=[early_stop_optuna, reduce_lr_optuna],
        verbose=1
    )

    best_val_accuracy = max(history.history["val_accuracy"])
    return best_val_accuracy

# 3.3. Esecuzione dello studio Optuna

Dopo aver definito la funzione `objective`, possiamo avviare lo studio di Optuna.

In questa fase la libreria proverà diverse configurazioni del modello, cambiando automaticamente gli iperparametri definiti in precedenza. Per ogni trial verrà costruita una CNN coerente con la nostra architettura di base, addestrata per un numero limitato di epoche e valutata sul validation set.

L’obiettivo è trovare la combinazione di parametri che massimizza la **validation accuracy**.

Poiché ogni trial richiede comunque un addestramento, il numero totale di prove va scelto con equilibrio:
- se i trial sono troppo pochi, la ricerca potrebbe essere superficiale;
- se sono troppi, il tuning rischia di diventare molto costoso in termini di tempo.

Un numero ragionevole può essere ad esempio **30 trial**.

In [ ]:
# Creazione dello studio Optuna
study = optuna.create_study(
    direction="maximize",
    study_name="cifar10_cnn_optimization",
    storage="sqlite:///optuna_cifar10.db",
    load_if_exists=True
)

study.optimize(objective, n_trials=20, catch=(tf.errors.InvalidArgumentError,))

print("Numero di trial completati:", len(study.trials))

# Considerazioni sull'ottimizzazione con Optuna

L'ottimizzazione automatica degli iperparametri tramite **Optuna** rappresenta, dal punto di vista teorico e metodologico, una strada molto valida per migliorare le prestazioni del modello. Questo approccio permette infatti di esplorare in modo sistematico diverse configurazioni della rete neurale, come numero di filtri, tassi di dropout, dimensione del livello denso e learning rate, con l'obiettivo di individuare la combinazione più efficace.

Nel progetto è stata quindi predisposta una pipeline compatibile con Optuna, integrata con il training del modello e con il salvataggio degli studi effettuati. Tuttavia, durante le prove pratiche è emerso un limite importante legato alle risorse hardware disponibili. Anche con la GPU correttamente configurata e utilizzata da TensorFlow, il costo computazionale dell'ottimizzazione è risultato molto elevato: già un singolo trial di poche epoche ha richiesto un tempo di esecuzione significativo.

Per questo motivo, pur essendo stata implementata e verificata la correttezza dell'impostazione, **l'ottimizzazione completa con Optuna non è stata portata avanti fino in fondo in questa fase del progetto**. Una ricerca estesa degli iperparametri richiede infatti hardware più performante, oppure tempi di esecuzione molto più lunghi, non compatibili con il contesto operativo attuale.

Di conseguenza, Optuna viene considerato in questo lavoro come **uno sviluppo metodologicamente corretto e potenzialmente molto utile**, ma da perseguire in modo concreto solo in presenza di risorse computazionali adeguate. In questa sede, i passi successivi dello studio proseguono quindi con un approccio più controllato e sostenibile, basato sull'analisi dei modelli costruiti e addestrati direttamente nel notebook.

## 3.4 Migliore configurazione trovata

Anche quando l'ottimizzazione non viene portata a termine con un numero elevato di trial, è comunque possibile leggere la migliore configurazione trovata tra quelle effettivamente eseguite.

Questo passaggio è importante perché permette di osservare:
- il miglior valore di validation accuracy ottenuto durante il tuning;
- i valori degli iperparametri associati a quella prestazione;
- la configurazione che, tra quelle provate, ha mostrato il comportamento più promettente.

In un contesto ideale, questi parametri verrebbero poi usati per ricostruire il modello finale da addestrare in modo completo. In questo progetto, il recupero della migliore configurazione consente comunque di documentare correttamente il funzionamento della procedura di tuning e di mostrare quali scelte architetturali siano risultate più favorevoli tra quelle esplorate.

In [ ]:
print("Miglior validation accuracy:", study.best_value)
print("\nMigliori iperparametri trovati:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

## 3.5 Analisi dei risultati del tuning

Oltre a leggere il best trial, è utile osservare come si sono distribuiti i risultati dei vari tentativi effettivamente completati.

Questo aiuta a capire se:
- Optuna ha trovato rapidamente una buona configurazione;
- i trial hanno prodotto risultati molto diversi tra loro;
- esistono configurazioni chiaramente peggiori o chiaramente migliori.

Nel primo grafico seguente viene rappresentata la validation accuracy ottenuta in ciascun trial completato.

Il secondo grafico mostra invece il **miglior valore raggiunto fino a ogni trial**. Questo tipo di visualizzazione è utile per capire se la ricerca stia portando un miglioramento progressivo oppure se i risultati si stabilizzino molto presto.

Se la curva cresce rapidamente e poi tende ad appiattirsi, significa che la ricerca ha individuato abbastanza presto una configurazione promettente. Se invece continua a salire anche nei trial successivi, vuol dire che lo spazio degli iperparametri contiene ancora combinazioni potenzialmente migliori da esplorare.

Nel nostro caso, l'interpretazione dei grafici deve tenere conto del numero limitato di trial eseguiti: l'analisi è comunque utile per documentare l'andamento del tuning, ma non consente di trarre conclusioni definitive sull'intero spazio di ricerca.

In [ ]:
# Accuracy di ogni trial
trial_values = [trial.value for trial in study.trials if trial.value is not None]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(trial_values) + 1), trial_values, marker="o")
plt.title("Validation accuracy nei trial di Optuna")
plt.xlabel("Trial")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.show()

# Best value cumulativo
best_so_far = []
current_best = -np.inf

for value in trial_values:
    if value > current_best:
        current_best = value
    best_so_far.append(current_best)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(best_so_far) + 1), best_so_far, marker="o")
plt.title("Miglior validation accuracy cumulativa durante Optuna")
plt.xlabel("Trial")
plt.ylabel("Best Validation Accuracy")
plt.grid(True)
plt.show()

# 4. Nuova base del modello migliorato

Dopo aver individuato tramite Optuna la configurazione più promettente, costruiamo una nuova versione del modello utilizzando i migliori iperparametri trovati durante il tuning.

La nuova architettura integra:
- **data augmentation migliorata**;
- **Batch Normalization**;
- **Dropout**;
- una struttura convoluzionale più solida, configurata sulla base dei risultati ottenuti da Optuna.

La data augmentation viene inserita all’inizio del modello, in modo che sia applicata solo durante il training.

Rispetto alla versione precedente, oltre alle trasformazioni geometriche già presenti, introduciamo anche leggere variazioni nello spazio colore, come:
- piccole modifiche di **luminosità**;
- leggere variazioni di **contrasto**;
- una lieve alterazione della **saturazione**.

L’obiettivo è rendere il modello meno sensibile a condizioni di luce e colore troppo specifiche del training set, migliorando la robustezza sulle classi più ambigue.

Questa sezione prepara quindi il modello finale da addestrare in modo completo utilizzando la migliore configurazione trovata durante il tuning.

In [ ]:
# 6. Nuova base del modello migliorato con i best params di Optuna

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    GlobalAveragePooling2D
)

# Recupero dei migliori iperparametri trovati
best_params = study.best_trial.params

print("Migliori iperparametri trovati da Optuna:")
for key, value in best_params.items():
    print(f"{key}: {value}")


# Layer custom per leggere variazioni di colore
class ColorJitter(tf.keras.layers.Layer):
    def __init__(self, brightness=0.08, contrast=0.12, saturation=0.10, **kwargs):
        super().__init__(**kwargs)
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation

    def call(self, images, training=None):
        if training is False:
            return images

        images = tf.image.random_brightness(images, max_delta=self.brightness)
        images = tf.image.random_contrast(
            images,
            lower=1.0 - self.contrast,
            upper=1.0 + self.contrast
        )
        images = tf.image.random_saturation(
            images,
            lower=1.0 - self.saturation,
            upper=1.0 + self.saturation
        )
        images = tf.clip_by_value(images, 0.0, 1.0)
        return images


# Data augmentation migliorata
data_augmentation_advanced = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    ColorJitter(brightness=0.08, contrast=0.12, saturation=0.10)
], name="data_augmentation_advanced")


def build_final_model_from_optuna(best_params, input_shape=(32, 32, 3), num_classes=10):
    model = Sequential([
        Input(shape=input_shape),

        data_augmentation_advanced,

        Conv2D(best_params["filters_1"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(best_params["filters_1"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(best_params["dropout_1"]),

        Conv2D(best_params["filters_2"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(best_params["filters_2"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(best_params["dropout_2"]),

        Conv2D(best_params["filters_3"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(best_params["filters_3"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(best_params["dropout_3"]),

        Conv2D(best_params["filters_4"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(best_params["filters_4"], (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(best_params["dropout_4"]),

        GlobalAveragePooling2D(),
        Dense(best_params["dense_units"], activation="relu"),
        BatchNormalization(),
        Dropout(best_params["dropout_dense"]),
        Dense(num_classes, activation="softmax")
    ], name="cifar10_final_model_optuna")

    return model


final_model = build_final_model_from_optuna(best_params)
final_model.summary()

# 5. Training del modello finale

Dopo aver costruito il modello finale utilizzando la configurazione migliore trovata durante il tuning, procediamo con l’addestramento completo.

Come già introdotto nel modello base, le callback ci aiutano a controllare meglio il training:
- **EarlyStopping** interrompe l’addestramento quando la validation loss non migliora più per un certo numero di epoche, evitando training inutilmente lunghi e riducendo il rischio di overfitting;
- **ReduceLROnPlateau** riduce automaticamente il learning rate quando il modello entra in una fase di stallo temporaneo, favorendo una convergenza più stabile.

In questo modo il modello può sfruttare un numero massimo elevato di epoche, mantenendo però un comportamento adattivo durante l’addestramento.

In [ ]:
# 7. Training del modello finale

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

final_model.compile(
    optimizer=Adam(learning_rate=best_params["learning_rate"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = final_model.fit(
    x_train,
    y_train,
    epochs=200,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# 6. Esportazione del modello finale

Una volta completato l’addestramento, il modello finale può essere salvato su file per essere riutilizzato in seguito senza dover ripetere il training.

In questo progetto utilizziamo il formato **`.keras`**, che rappresenta il formato nativo di salvataggio di Keras. Questo permette di conservare in un unico file:
- l’architettura del modello;
- i pesi appresi durante l’addestramento;
- la configurazione necessaria per il riutilizzo del modello.

Il salvataggio del modello è utile sia per future analisi, sia per un’eventuale integrazione in un’applicazione esterna o in una piccola demo di presentazione del progetto.

In [ ]:
import os
from datetime import datetime
from pathlib import Path
import tensorflow as tf

# Definizione delle variabili di percorso
PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODEL_EXPORT_PATH = os.path.join(PROJECT_ROOT, "RESULT")

def export_keras_model(model, base_path=MODEL_EXPORT_PATH):
    """
    Esporta un modello Keras in formato .keras (standard) 
    e in formato SavedModel (per il deployment).
    """
    # 1. Genera un timestamp per il versioning
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_dir = os.path.join(base_path, f"model_{timestamp}")
    
    # 2. Crea la struttura delle cartelle
    try:
        os.makedirs(export_dir, exist_ok=True)
        print(f"--- Inizio esportazione in: {export_dir} ---")
        
        # 3. Salva in formato nativo Keras (singolo file h5/keras)
        # Ideale per ricaricare il modello in Python per continuare il training
        native_path = os.path.join(export_dir, "cifar10_improved_model_V2.keras")
        model.save(native_path)
        print(f"[OK] Modello Keras salvato: {native_path}")
        
        # 4. Salva in formato SavedModel (cartella con asset e variabili)
        # Standard per TensorFlow Serving, TFLite e TensorFlow.js
        sm_path = os.path.join(export_dir, "saved_model")
        model.save(sm_path, save_format="tf")
        print(f"[OK] SavedModel pronto per il deployment: {sm_path}")
        
        return export_dir

    except Exception as e:
        print(f"[ERRORE] Esportazione fallita: {e}")
        return None

# Esempio di utilizzo (assicurati che 'mio_modello' sia definito):
export_keras_model(final_model)